# 第07课：工具调用与Agent基础
上一课是“程序先找资料，再带原文问模型”。这次多一个选择：**模型可以提出“请查这个工具”的请求，但真正运行函数的是应用程序，不是模型。**

本课只读虚构图书馆数据，不访问个人文件，不写数据库，不执行Shell，不引入Agent框架。主线是看懂一次请求的去向，不再练余弦或排序算法。

本课结束时要能：①拒绝未知/未授权工具，安全处理执行结果；②把工具结果按调用ID交回模型；③连接有限循环，能直接结束、处理两步依赖、遇到重复或预算到达时停止。教师验证不是学习者已掌握的证据。

## 0. 先看完整时间线
| 时刻 | 谁在做 | 数据是什么 | 是否调用聊天API |
|---|---|---|---|
| T0 | 开发者准备 | 工具说明tools、真实函数registry、执行白名单 | 否 |
| T1 | 程序问模型 | system规则、user问题、工具说明 | 是，第1次 |
| T2 | 模型返回 | assistant工具调用请求，含名称/参数/ID；尚未执行 | 无新增请求 |
| T3 | 程序决定并执行 | 查白名单→校验参数→调用本地函数，得到字典 | 本地工具不调用聊天API |
| T4 | 程序准备下一次消息 | 保留assistant请求，再放role=tool结果并配对ID | 否 |
| T5 | 程序再次问模型 | 原问题、原调用、工具结果 | 是，第2次 |
| T6 | 模型返回 | 最终文字，或新的工具请求 | 需要更多工具时才进入下一步 |

**“一步”在本课指问模型一次，不是一轮user/assistant，也不是执行一个工具。** 一个模型响应可提出多个工具请求。本课顺序执行，不研究并发。

Agent可先理解为“模型＋工具＋推进/停止规则”的应用，不是新模型。RAG检索也可封装成工具，但本课先用最短只读查询，不要求向量数据库。[OpenAI官方五步流程](https://developers.openai.com/api/docs/guides/function-calling) · [DeepSeek工具调用](https://api-docs.deepseek.com/guides/tool_calls/)

HTML是短复习：[微课1](../lessons/0010-tools-map.html) · [微课2](../lessons/0011-tools-return.html) · [微课3](../lessons/0012-agent-loop.html)。不清楚时直接在聊天报小节编号，不必独自硬读全部。

## 运行本课前：用哪一个Python？
本课不需要FastEmbed，不下载新模型。可沿用第6课已可运行Notebook的环境，也可用已装本课依赖的其他内核；**不要只凭环境都叫.venv就认为是同一个**。

在本课目录打开终端，Windows PowerShell、macOS/Linux均可：
```sh
python -m pip install -r requirements.txt
python -m pip show openai python-dotenv pytest
```
python是所选解释器；-m pip用它的包管理器；install安装；-r读取清单。openai负责兼容SDK接线（实际服务是DeepSeek），python-dotenv只读本地配置，pytest跑离线测试。范围沿用已验证OpenAI SDK 2.54.0，兼容Python 3.12/3.14。

Notebook若仍提示缺包，在**当前Notebook新格**运行：
```python
%pip install -r requirements.txt
```
该命令要求Notebook工作目录为本课目录；必要时重启内核，再运行准备格。终端装过不等于当前内核装过。用`import sys; print(sys.version.split()[0])`核对版本；解释器路径可在本机查看，但不要保存含用户名的输出进仓库。无需为了本课再建一套虚拟环境。

真实请求开关在第2.4节，默认False；默认从头运行不会下载、联网或计费。

In [1]:
from pathlib import Path
from copy import deepcopy
import sys
import json

_candidates = [
    Path.cwd(),
    Path.cwd() / "03_大模型应用开发" / "07_工具调用与Agent基础",
]
COURSE_DIR = next((path for path in _candidates if (path / "tool_support.py").is_file()), None)
if COURSE_DIR is None:
    print("❌ 找不到本课文件；请在本课目录或仓库根目录启动Notebook。")
else:
    if str(COURSE_DIR) not in sys.path:
        sys.path.insert(0, str(COURSE_DIR))
    from tool_support import (
        AGENT_RULE, LIBRARY_TOOLS, LIBRARY_REGISTRY, LIBRARY_ALLOWED,
        find_branch, get_opening, sample_call, validate_arguments,
        operation_key, observe_roundtrip,
    )
    print("✅ 准备完成；导入教师模块不联网。")

✅ 准备完成；导入教师模块不联网。


准备格只做路径查找和导入，沿用第6课写法。deepcopy来自标准库copy，稍后在2.1专门对比嵌套复制。tool_support里的名字均为**教师定义**，不是Python关键字；本课用到的教师函数都在首次调用前说明输入、输出和时刻。

## 1. 模型提出请求，程序才有机会执行
### 1.1 工具说明、真实函数、执行许可，是三样东西
以“查lib-east分馆周一是否开放”为例：
- **工具说明tools**：开发者写的list[dict]，告诉模型有一个get_opening，接受branch_id字符串。只是菜单，没有把Python函数本体传给模型。
- **真实函数**：教师写的get_opening(branch_id: str) → dict，读取本机固定教学数据。
- **registry**：开发者准备的dict，把工具名称str映射到Python函数。值是函数本身，不加括号。
- **allowed_names / 执行白名单**：程序本次获准执行的名称set[str]。注册过不等于每次都获准。

下面构造或查看说明不会执行工具。参数Schema沿用第3课：properties写字段，required列必填字段；additionalProperties=False声明不接受额外字段。普通非strict调用仍需本地校验，不把Schema当执行授权。

In [9]:
print("给模型的工具说明：")
print(json.dumps(LIBRARY_TOOLS[1], ensure_ascii=False, indent=2))
print("已注册名称：", list(LIBRARY_REGISTRY))
print("本次获准名称：", sorted(LIBRARY_ALLOWED))
# 上面都没有调用get_opening。下面这一行才实际执行本地查询：
print("真实函数返回：", get_opening("lib-east"))

给模型的工具说明：
{
  "type": "function",
  "function": {
    "name": "get_opening",
    "description": "按分馆编号查询开放日；不知道编号时先查分馆。只读。",
    "parameters": {
      "type": "object",
      "properties": {
        "branch_id": {
          "type": "string",
          "description": "分馆编号"
        }
      },
      "required": [
        "branch_id"
      ],
      "additionalProperties": false
    }
  }
}
已注册名称： ['find_branch', 'get_opening']
本次获准名称： ['find_branch', 'get_opening']
真实函数返回： {'found': True, 'branch_id': 'lib-east', 'open_days': '周二至周日', 'closed': '周一'}


LIBRARY_TOOLS[1]是第二条工具说明；列表索引从0开始。json.dumps的ensure_ascii=False保留中文，indent=2让输出便于阅读；返回值仍是str。sorted这里只让集合输出稳定，不是本课的检索算法。

get_opening("lib-east")给教师函数传入一个位置参数branch_id，返回dict，例如found=True、closed="周一"。换成未知编号也会返回正常查询结果found=False，**不等于函数执行失败**。

In [3]:
print("换一个编号：", get_opening("lib-missing"))
proposal_name = "delete_branch"
print("模型即使提出这个名称，是否在注册表？", proposal_name in LIBRARY_REGISTRY)
print("是否获准？", proposal_name in LIBRARY_ALLOWED)
print("未注册/未授权就不运行；没有任何删除动作。")

换一个编号： {'found': False, 'branch_id': 'lib-missing'}
模型即使提出这个名称，是否在注册表？ False
是否获准？ False
未注册/未授权就不运行；没有任何删除动作。


### 1.2 模型返回的调用请求长什么样？
sample_call(call_id: str, name: str, arguments: dict) → dict是教师**生成固定演示数据**的函数，三个位置参数分别是ID、名称、参数字典；它会序列化arguments。它没有请求模型，下面不冒充真实模型输出。

| 字段 | 提供者与位置 | 类型/含义 |
|---|---|---|
| tools里的function.name/parameters | 开发者，发送给模型 | 工具名称str与参数规则dict |
| tool_calls | 服务端assistant响应 | 调用字典列表，不是工具结果 |
| id | 服务端调用请求中返回 | str，这一次调用的配对编号，不是RAG资料ID |
| function.name | 服务端调用请求中返回 | str，模型想调用哪个工具 |
| function.arguments | 服务端调用请求中返回 | **JSON字符串**，不是Python字典 |
| tool_call_id | 程序构造tool消息时复制 | str，必须等于所回答调用的id |

这些字段名来自Chat Completions协议，不是Python关键字；id由响应提供，程序不能随便新造一个替换它。[DeepSeek字段说明](https://api-docs.deepseek.com/api/create-chat-completion/)

In [4]:
demo_call = sample_call("call-open", "get_opening", {"branch_id": "lib-east"})
print(demo_call)
print("名称：", demo_call["function"]["name"])
print("参数文本：", demo_call["function"]["arguments"])
print("参数类型：", type(demo_call["function"]["arguments"]).__name__)
print("到这里依然只是一个请求字典，函数还没有因它而自动运行。")

{'id': 'call-open', 'type': 'function', 'function': {'name': 'get_opening', 'arguments': '{"branch_id": "lib-east"}'}}
名称： get_opening
参数文本： {"branch_id": "lib-east"}
参数类型： str
到这里依然只是一个请求字典，函数还没有因它而自动运行。


### 1.3 从请求到执行：先许可，再参数，再函数
validate_arguments(name: str, raw: str) → dict是教师函数：只接受本课工具定义的必填字符串字段，不符合时抛ValueError("bad_arguments")；把通用类型/长度/JSON校验留在教师底层，不让它成为本课作业主体。None不是本课合法参数；不得跳过校验直接运行。

下面只处理已经知道且获准的演示名称get_opening。registry[name]取到函数；**args是Python字典的关键字参数解包**，相当于get_opening(branch_id="lib-east")，不是SDK魔法，也不是执行JSON代码。

In [5]:
demo_name = demo_call["function"]["name"]
if demo_name not in LIBRARY_REGISTRY:
    print("拒绝：未知工具。")
elif demo_name not in LIBRARY_ALLOWED:
    print("拒绝：未获准。")
else:
    demo_args = validate_arguments(demo_name, demo_call["function"]["arguments"])
    demo_data = LIBRARY_REGISTRY[demo_name](**demo_args)
    demo_outcome = {"ok": True, "data": demo_data}
    print("程序实际执行后的结果：", demo_outcome)

程序实际执行后的结果： {'ok': True, 'data': {'found': True, 'branch_id': 'lib-east', 'open_days': '周二至周日', 'closed': '周一'}}


ok/data/error是**本课自定义工具结果字段**，不是HTTP状态码或API固定字段。正常查询返回{"ok": True, "data": 工具返回值}；即使data里found=False，也表示查询成功但没有业务记录。

未知工具/无权执行不能调用函数。参数坏了就拒绝执行。实际工具TimeoutError与其他异常分别回传tool_timeout/tool_failed，不把执行失败说成“没有这条记录”，也不把异常全文发给模型。真实写入需要单独授权和幂等设计；幂等在这里指相同请求重复到达时，不造成重复扣款等额外副作用。本课直接禁止写入，不提供“让模型批准自己”的选项。

下面是坏参数反例：先验参数，因此不会运行函数。

In [6]:
try:
    validate_arguments("get_opening", '{"branch_id":"lib-east","command":"delete"}')
except ValueError:
    print("❌ 参数有额外字段；在调用真实函数之前阻止。")

❌ 参数有额外字段；在调用真实函数之前阻止。


### 课堂练习1：判断园区工具有没有执行许可
目标：区分“模型提出”“注册过”“获准执行”。**只填字典，不写参数校验函数。**

已注册：query_gate（只读查询）、delete_gate（修改数据）。本次白名单只有query_gate。a模型请求query_gate；b请求delete_gate；c请求send_mail（未注册）。
将garden_access的a/b/c分别填写为三个字符串之一：allowed（已注册且获准）、not_allowed（已注册但未获准）、unknown_tool（未注册）。分类名称与后续工具错误码保持一致。预期：三个场景都被准确分类；不真的执行任何工具。

In [9]:
garden_access = {"a": "allowed", "b": "not_allowed", "c": "unknown_tool"}

In [10]:
try:
    assert isinstance(garden_access, dict), "garden_access应为a/b/c三个分类值的字典。"
    for _check_key, _check_expected, _check_hint in [
        ("a", "allowed", "a：query_gate已注册且在本次白名单，应允许进入参数校验。"),
        ("b", "not_allowed", "b：delete_gate虽注册过，但本次无写入授权。"),
        ("c", "unknown_tool", "c：send_mail没有注册，不能按名称寻找任意Python函数。"),
    ]:
        assert garden_access.get(_check_key), f"先填写场景{_check_key}。"
        assert garden_access[_check_key] == _check_expected, _check_hint
    print("✅ 课堂练习1：模型请求与执行许可已分开判断。")
except NameError:
    print("❌ 先运行紧上方garden_access答题格。")
except AssertionError as _check_error:
    print("❌", _check_error)
except Exception as _check_error:
    print("❌ 权限分类结构不符：", type(_check_error).__name__, "；只填a/b/c的分类字符串。")

✅ 课堂练习1：模型请求与执行许可已分开判断。


<details><summary>独立完成后查看参考要点</summary>

a填allowed，b填not_allowed，c填unknown_tool。模型想做不等于程序能做；注册目录也不等于本次许可。后续作业把这一判断用于阻止回调执行。

</details>

## 2. 把结果交回模型：它不知道本地刚查到了什么
到T3，程序执行函数获得demo_outcome。但模型不会自动看见你的局部变量。**要让它使用结果，还需要把结果作为消息发回去。**

### 2.1 先补一个副本知识点
第四课用过列表/消息字典复制。本课assistant字典里又包含tool_calls列表、function字典，所以只做message.copy()还不够。deepcopy(obj)来自copy标准库，接收一个位置参数，返回包含这些嵌套字典/列表的独立副本；本课只复制JSON形状的数据，不复制客户端对象。

【先预测，再运行】仅对每条消息做浅复制后，修改副本内tool_calls的id，原消息会不会跟着变？先填prediction_nested（bool），再运行检测与下面演示。[Python复制说明](https://docs.python.org/3/library/copy.html)

In [11]:
prediction_nested = True  # 填True或False：原消息是否跟着变？

In [12]:
try:
    assert type(prediction_nested) is bool, "先把prediction_nested填为True或False。"
    assert prediction_nested is True, "浅复制只复制外层消息，嵌套tool_calls仍共享。"
    print("✅ 预测通过：嵌套列表仍共享，下面用deepcopy对照。")
except NameError:
    print("❌ 先运行prediction_nested答题格。")
except AssertionError as _check_error:
    print("❌", _check_error)
except Exception as _check_error:
    print("❌ 预测值应为bool：", type(_check_error).__name__)

✅ 预测通过：嵌套列表仍共享，下面用deepcopy对照。


In [13]:
# 这里只保留嵌套id字段来观察复制，不是完整API消息。
original = [{"tool_calls": [{"id": "call-a"}]}]
shallow = [row.copy() for row in original]
shallow[0]["tool_calls"][0]["id"] = "changed"
print("浅复制后原数据：", original[0]["tool_calls"][0]["id"])

original = [{"tool_calls": [{"id": "call-a"}]}]
isolated = deepcopy(original)
isolated[0]["tool_calls"][0]["id"] = "changed"
print("深复制后原数据：", original[0]["tool_calls"][0]["id"])

浅复制后原数据： changed
深复制后原数据： call-a


上面第一行输出changed，第二行输出call-a。deepcopy解决请求快照共享；安全沙箱是限制代码可访问资源的隔离环境，复制数据并不提供这种隔离。第四课旧答案不在这里自动改写。

### 2.2 两条消息，一定按顺序放
先保留模型原来的assistant工具请求，再追加程序的tool结果。不能只发结果，否则缺少“这次结果回答哪一个调用”的协议关系。也不能把结果放进system当规则。

正常消息顺序是：system → user → assistant(tool_calls) → tool(tool_call_id) → 再请求模型。tool消息的content要是**字符串**；本课用json.dumps把结果字典转成JSON文本。工具失败也要有匹配的tool错误结果，避免只回成功的那一半。

下面接续第1节demo_call/demo_outcome；这里只构造消息，没有发送请求。

In [14]:
base_messages = [
    {"role": "system", "content": AGENT_RULE},
    {"role": "user", "content": "lib-east周一开放吗？"},
]
assistant_request = {"role": "assistant", "content": None, "tool_calls": [demo_call]}

round_messages = deepcopy(base_messages)
round_messages.append(deepcopy(assistant_request))
round_messages.append({
    "role": "tool",
    "tool_call_id": demo_call["id"],
    "content": json.dumps(demo_outcome, ensure_ascii=False),
})
print("消息角色顺序：", [row["role"] for row in round_messages])
print("工具结果消息：", round_messages[-1])
print("原base_messages长度仍为：", len(base_messages))

消息角色顺序： ['system', 'user', 'assistant', 'tool']
工具结果消息： {'role': 'tool', 'tool_call_id': 'call-open', 'content': '{"ok": true, "data": {"found": true, "branch_id": "lib-east", "open_days": "周二至周日", "closed": "周一"}}'}
原base_messages长度仍为： 2


assistant_request的content=None在“有工具调用”时可以正常出现，不代表请求失败；调用内容在tool_calls里。第06课的None正文检查不能不分场景直接搬过来。

tool_call_id必须复制demo_call["id"]。相同工具被请求两次，也有两个不同的调用ID；工具名不能充当配对编号。一个响应有多个调用时，**先保留这条assistant，再为每个调用回一条tool消息，全部完成后再问模型**。本课不使用只处理tool_calls[0]的通用循环；上面的单条只为看清最小关系。

例如：call-a查lib-east，call-b查lib-west，结果需要分别标call-a/call-b，不能随便调换内容。即使其中一个返回{"ok": False, "error": "tool_timeout"}，它也要回答自己的ID，不能丢掉。[官方配对协议](https://api-docs.deepseek.com/guides/tool_calls/)

### 课堂练习2：构造一条园区工具结果消息
目标：把Python结果字典包装成协议要求的tool消息，不联网。
输入：
- call_id：str，例如"gate-call-1"，来自模型调用请求的id。
- result：dict，例如{"ok": True, "data": {"gate": "东门"}}，来自程序执行结果。

完成garden_tool_message(call_id, result)，返回一个**新字典**，恰好有role、tool_call_id、content三个键。role固定为"tool"；tool_call_id保留输入ID；content是result的JSON字符串，保留可读中文。不能修改result，也不能把result字典直接塞进content。错误结果也用相同包装。
这里只返回单条消息，完整消息列表组装是作业函数2。只补函数体，不套同名def。

In [17]:
def garden_tool_message(call_id: str, result: dict) -> dict:
    # TODO：返回一条按ID配对的tool消息，content为JSON字符串
    return {
        'role': 'tool',
        'tool_call_id': call_id,
        'content': json.dumps(result, ensure_ascii=False)
    }

In [18]:
try:
    assert callable(garden_tool_message), "先定义garden_tool_message。"
    _check_input = {"ok": True, "data": {"gate": "东门"}}
    _check_before = deepcopy(_check_input)
    _check_message = garden_tool_message("gate-call-1", _check_input)
    assert isinstance(_check_message, dict), "正常场景：应返回消息字典，不是None或列表。"
    assert set(_check_message) == {"role", "tool_call_id", "content"}, "字段场景：只含role/tool_call_id/content。"
    assert _check_message["role"] == "tool", "角色场景：工具结果不是system、user或assistant。"
    assert _check_message["tool_call_id"] == "gate-call-1", "配对场景：必须原样保留模型给的调用ID。"
    assert isinstance(_check_message["content"], str), "正文场景：content必须是字符串，不能直接放字典。"
    assert json.loads(_check_message["content"]) == _check_input, "数据场景：序列化后不能漏掉或改写工具结果。"
    assert "东门" in _check_message["content"], "中文场景：序列化时保留可读中文。"
    assert _check_input == _check_before, "副本场景：不能修改输入result。"
    _check_error_message = garden_tool_message("gate-call-2", {"ok": False, "error": "tool_timeout"})
    assert _check_error_message["tool_call_id"] == "gate-call-2", "变化场景：不同调用ID不能写死。"
    assert json.loads(_check_error_message["content"]) == {"ok": False, "error": "tool_timeout"}, "失败也要回传原错误结果。"
    print("✅ 课堂练习2：结果正文与调用ID配对通过。")
except NameError:
    print("❌ 先运行准备格与garden_tool_message答题格。")
except AssertionError as _check_error:
    print("❌", _check_error)
except json.JSONDecodeError:
    print("❌ content不是合法JSON文本；序列化结果字典，不手工拼字符串。")
except (KeyError, TypeError) as _check_error:
    print("❌ 消息字段或类型不符：", type(_check_error).__name__, "；对照题目三个字段。")
except Exception as _check_error:
    print("❌ 运行异常：", type(_check_error).__name__, "；检查返回消息，不修改检测代码。")

✅ 课堂练习2：结果正文与调用ID配对通过。


<details><summary>独立完成后查看参考答案</summary>

```python
def garden_tool_message(call_id, result):
    return {
        "role": "tool",
        "tool_call_id": call_id,
        "content": json.dumps(result, ensure_ascii=False),
    }
```

这里只打包结果；不会调用API，不负责判断工具结果里的文字是否真实或安全。

</details>

### 2.3 谁把SDK对象变成字典？
SDK返回的message是SDK对象，可能用message.tool_calls访问；本课学生代码拿到的是**教师适配后的字典**，必须用message["tool_calls"]。二者不要混用。

tool_support.normalize_response(response) → dict由教师提供：只接受finish_reason="tool_calls"且有完整调用，或finish_reason="stop"且有非空最终文字；length等未完成响应不交给执行器。它还检查同批调用ID不重复。**工具请求不是最终回答**，所以tool_calls结束原因不能当成stop。

create_client(folder: Path)是教师函数，返回(client, model)：client是用第三方openai库的OpenAI类创建的客户端实例，指向DeepSeek；model是配置中的模型名字符串。创建对象不是发送请求。

request_assistant(client, model, messages, tools, tool_choice="auto")是教师单次接线函数，前四项为位置参数；返回(assistant字典, usage字典或None)。底层由client.chat.completions.create真正请求：
- model是服务端模型名str；messages是本次完整消息列表；tools是工具说明列表，不是Python函数注册表。
- tool_choice="auto"允许模型决定是否用工具；"none"禁止本次新调用；指定名称的字典要求使用某工具。本课不传None；省略时教师包装默认auto。
- max_tokens=240限制输出token，不是字数；无stream=True所以非流式。
- extra_body={"thinking":{"type":"disabled"}}是DeepSeek扩展，用来关闭思考以聚焦本课协议。不要据此推断其他服务商参数相同。
- 客户端设置timeout=30、max_retries=0，零自动重试。超时不是资料不足；不在学生代码里重新抄SDK、密钥查找或异常类型清单。

本课不用Beta strict接口，也不假设模型参数一定合法。若开启思考模式，DeepSeek还有reasoning_content回传要求，超出本课范围，不要只改开关就沿用当前适配。[工具接口](https://api-docs.deepseek.com/guides/tool_calls/) · [思考接口](https://api-docs.deepseek.com/guides/thinking_mode/)

### 2.4 可选真实观察：最多两次短请求
先看[真实工具观察记录](真实工具观察记录.md)。运行下格才可能联网、计费；默认False。无需模型下载，可复用第01课本地.env，勿把密钥粘进Notebook。

observe_roundtrip(folder: Path)是教师固定接线：第1次**指定get_opening**以观察调用格式→仅执行这一个获准的本地只读查询→第2次设tool_choice="none"观察回答。返回包含model/messages/usage/model_calls/tool_calls的dict；客户端在finally关闭。这是协议实验，不是自主选择工具或完整Agent评估。错误时不自动重跑；没有usage不能推算实测费用。

In [19]:
RUN_REAL_TOOLS = False  # 手动开启；运行一次最多发送两次短请求
live_observation = None

if not RUN_REAL_TOOLS:
    print("⏭️ 未发送真实请求。可先读真实工具观察记录，再继续第3节。")
else:
    try:
        live_observation = observe_roundtrip(COURSE_DIR)
        print("本次模型：", live_observation["model"])
        print("模型请求次数：", live_observation["model_calls"])
        print("本地工具执行次数：", live_observation["tool_calls"])
        for live_message in live_observation["messages"][2:]:
            print(live_message)
        print("usage：", live_observation["usage"])
    except ValueError:
        print("❌ 配置或协议检查未通过；检查本地配置和教师接线，不自动重试。")
    except Exception as live_error:
        print("❌ 真实观察未完成：", type(live_error).__name__, "；不回显异常原文、不自动重试。")

本次模型： deepseek-flash
模型请求次数： 2
本地工具执行次数： 1
{'role': 'assistant', 'content': '', 'tool_calls': [{'id': 'call_00_Q3vKaqk30tm5GkveM4vs3621', 'type': 'function', 'function': {'name': 'get_opening', 'arguments': '{"branch_id": "lib-east"}'}}]}
{'role': 'tool', 'tool_call_id': 'call_00_Q3vKaqk30tm5GkveM4vs3621', 'content': '{"ok": true, "data": {"found": true, "branch_id": "lib-east", "open_days": "周二至周日", "closed": "周一"}}'}
{'role': 'assistant', 'content': 'lib-east 周一不开放，开放日为周二至周日。', 'tool_calls': []}
usage： [{'completion_tokens': 35, 'prompt_tokens': 383, 'total_tokens': 418, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 128}, 'prompt_cache_hit_tokens': 128, 'prompt_cache_miss_tokens': 255}, {'completion_tokens': 13, 'prompt_tokens': 180, 'total_tokens': 193, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_t

## 3. Agent循环：查完一次不一定够，但必须能停
### 3.1 两步依赖，不是一次响应里乱猜参数
问题变为“云城图书馆周一开放吗？”用户没给分馆编号。顺序应是：
1. 模型请求find_branch(city="云城")；程序查到branch_id="lib-east"。
2. 将结果回给模型后，模型才请求get_opening(branch_id="lib-east")。
3. 再回传开放日，模型给最终回答。

需要**3次模型请求＋2次本地工具处理**，不是“2个工具所以只问模型2次”。下面用固定假模型逐格走，不联网，也不宣称验证了真实模型自主规划。

demo_ask(messages: list[dict]) → dict是本格定义的普通函数。它记录收到的消息，按次数返回预写好的响应；不是SDK。messages是传入的消息列表。每次deepcopy是保存独立快照，避免后面append把早先记录也改了。

In [21]:
scripted_replies = [
    {"role": "assistant", "content": None,
     "tool_calls": [sample_call("call-branch", "find_branch", {"city": "云城"})]},
    {"role": "assistant", "content": None,
     "tool_calls": [sample_call("call-days", "get_opening", {"branch_id": "lib-east"})]},
    {"role": "assistant", "content": "云城东馆周一闭馆，开放日为周二至周日。", "tool_calls": []},
]
demo_model_seen = []

def demo_ask(messages: list[dict]) -> dict:
    demo_model_seen.append(deepcopy(messages))
    return deepcopy(scripted_replies[len(demo_model_seen) - 1])

manual_history = [
    {"role": "system", "content": AGENT_RULE},
    {"role": "user", "content": "云城图书馆周一开放吗？"},
]
first_reply = demo_ask(manual_history)
print("第一次假模型返回：", first_reply)

第一次假模型返回： {'role': 'assistant', 'content': None, 'tool_calls': [{'id': 'call-branch', 'type': 'function', 'function': {'name': 'find_branch', 'arguments': '{"city": "云城"}'}}]}


现在只拿到find_branch请求，还没查数据。为了看清时刻，下面手动完成这一个调用；固定示例已知只有一个请求，因此取[0]。这不是可以处理任意响应的通用Agent。

In [23]:
first_call = first_reply["tool_calls"][0]
first_name = first_call["function"]["name"]
assert first_name in LIBRARY_ALLOWED and first_name in LIBRARY_REGISTRY
first_args = validate_arguments(first_name, first_call["function"]["arguments"])
first_outcome = {"ok": True, "data": LIBRARY_REGISTRY[first_name](**first_args)}
manual_history.append(deepcopy(first_reply))
manual_history.append({
    "role": "tool", "tool_call_id": first_call["id"],
    "content": json.dumps(first_outcome, ensure_ascii=False),
})
print("第一次工具结果：", first_outcome)
print("此时模型调用次数：", len(demo_model_seen))

第一次工具结果： {'ok': True, 'data': {'found': True, 'branch_id': 'lib-east', 'name': '云城东馆'}}
此时模型调用次数： 1


查到了lib-east，但最终答案还没生成。再把完整历史交给假模型，它的下一条固定响应使用这个分馆编号。

In [24]:
second_reply = demo_ask(manual_history)
print("第二次假模型返回：", second_reply)
second_call = second_reply["tool_calls"][0]
second_name = second_call["function"]["name"]
assert second_name in LIBRARY_ALLOWED and second_name in LIBRARY_REGISTRY
second_args = validate_arguments(second_name, second_call["function"]["arguments"])
second_outcome = {"ok": True, "data": LIBRARY_REGISTRY[second_name](**second_args)}
manual_history.append(deepcopy(second_reply))
manual_history.append({
    "role": "tool", "tool_call_id": second_call["id"],
    "content": json.dumps(second_outcome, ensure_ascii=False),
})
final_reply = demo_ask(manual_history)
manual_history.append(deepcopy(final_reply))
print("最终文字：", final_reply["content"])
print("模型请求次数：", len(demo_model_seen), "；本地工具处理次数：2")
print("最终消息角色：", [row["role"] for row in manual_history])

第二次假模型返回： {'role': 'assistant', 'content': None, 'tool_calls': [{'id': 'call-days', 'type': 'function', 'function': {'name': 'get_opening', 'arguments': '{"branch_id": "lib-east"}'}}]}
最终文字： 云城东馆周一闭馆，开放日为周二至周日。
模型请求次数： 3 ；本地工具处理次数：2
最终消息角色： ['system', 'user', 'assistant', 'tool', 'assistant', 'tool', 'assistant']


最终角色是system/user/assistant/tool/assistant/tool/assistant。如果模型第一次就只返回普通文字（tool_calls=[]），应该直接结束、一次工具也不执行；不是每个问题都必须用工具。如果同一次响应里请求两个**相互独立**的工具，应在同一条assistant后回两条配对tool，再问一次模型，不是看到第一条成功就丢掉另一条。

### 3.2 给循环设上限：max_steps数的是模型调用
不能直接写“模型要工具就永远继续”。本课max_steps是**本次任务最多调用ask的次数**，非负整数；默认4，显式传2就只有2次，传0零次请求。None不是合法预算；不是tool_choice，不是token预算，也不是历史轮次。

本课采用一个明确策略：**最后一次模型调用如果仍提出工具请求，就不再启动这批工具，也不提交缺结果的半轮消息**。因为已经没有预算把新结果交给模型使用。正常最终文字即使在最后一次返回，仍然接受。

| max_steps | 模型返回经过 | 结果 |
|---|---|---|
| 0 | 不调用ask | step_limit，0次模型/0次工具 |
| 1 | 第1次直接最终文字 | complete，1次模型/0次工具 |
| 1 | 第1次提出工具 | step_limit，1次模型/0次工具 |
| 2 | 第1次工具，第2次最终文字 | complete，2次模型/1次工具 |
| 2 | 第1次工具，第2次又要新工具 | step_limit，2次模型/只处理第1批工具 |

这个策略是本课约定，不是SDK自动帮你做。max_steps只限制次数，不能强制终止卡住的任意Python函数；真实模型有请求超时，真实工具也必须自己实现超时。捕获TimeoutError不是沙箱或进程终止器。

### 3.3 重复调用：新ID不一定是新操作
模型可能先请求：
```text
call-1：get_opening({"branch_id":"lib-east"})
```
收到结果后又请求：
```text
call-2：get_opening({ "branch_id" : "lib-east" })
```
ID变了、JSON空格变了，但工具名与参数内容没变。本课停止为repeated_call，不再次处理这批请求。

operation_key(call: dict) → str是教师函数：从工具名＋参数的规范JSON生成“操作签名”，忽略调用ID、对象键顺序和JSON排版空格。它不执行工具；参数坏时仍给签名，具体拒绝由validate_arguments负责。字符串值内部空格属于参数内容，不会被随意删除。sort_keys=True只是让字典键顺序稳定，不是业务参数校验。

不同参数，如lib-east与lib-west，是两个不同操作，不应一概阻止。同一响应里两次相同操作也整批阻止，**任何一个工具都不先跑**。跨轮曾处理过的请求（包括被拒绝或失败的请求）也计入签名记录；这不是写操作的可靠幂等保证，跨进程恢复另学。

In [25]:
repeat_a = sample_call("call-1", "get_opening", {"branch_id": "lib-east"})
repeat_b = sample_call("call-2", "get_opening", {})
repeat_b["function"]["arguments"] = '{ "branch_id" : "lib-east" }'
different = sample_call("call-3", "get_opening", {"branch_id": "lib-west"})
print("不同ID、同参数是否同一操作：", operation_key(repeat_a) == operation_key(repeat_b))
print("不同参数是否同一操作：", operation_key(repeat_a) == operation_key(different))

不同ID、同参数是否同一操作： True
不同参数是否同一操作： False


本课循环最终status是程序定义的complete、step_limit、repeated_call。complete只表示模型返回最终文字，不保证事实正确，也不表示工具全成功；模型可以最终解释“工具超时，未拿到数据”。它不是第04课流收集器的complete语义，也不是HTTP状态码。

工具拒绝/失败由execute_call变成ok=False结果回传，模型可以据此说明限制；**模型ask本身抛异常则向调用方传播，不改成不知道、不自动重试**，沿用第06课的区别。协议不完整也由教师适配报错，不执行半截参数。

### 课堂练习3：判断园区循环何时停止
只填garden_stops的三个值，候选字符串为complete、step_limit、repeated_call。
- a：max_steps=1，第1次模型直接给非空最终文字，无工具请求。
- b：max_steps=1，第1次模型提出查入口请求；预算已用完，不启动工具。
- c：max_steps=4，第1次调用find_gate查园区，已处理并回传；第2次换新ID又请求同名同参数。

目标是区分正常结束、预算用尽和重复，不写基础循环。预期三个场景逐项通过。

In [26]:
garden_stops = {"a": "complete", "b": "step_limit", "c": "repeated_call"}

In [27]:
try:
    assert isinstance(garden_stops, dict), "garden_stops应为a/b/c三个停止状态的字典。"
    for _check_key, _check_expected, _check_hint in [
        ("a", "complete", "a：最后一次返回最终文字也正常结束，不是一律预算失败。"),
        ("b", "step_limit", "b：最后一次仍要工具，预算不足以使用新结果，本课不启动工具。"),
        ("c", "repeated_call", "c：新ID不改变同名同参数的重复操作。"),
    ]:
        assert garden_stops.get(_check_key), f"先填写场景{_check_key}。"
        assert garden_stops[_check_key] == _check_expected, _check_hint
    print("✅ 课堂练习3：停止原因与执行时刻通过。")
except NameError:
    print("❌ 先运行garden_stops答题格。")
except AssertionError as _check_error:
    print("❌", _check_error)
except Exception as _check_error:
    print("❌ 停止状态结构不符：", type(_check_error).__name__, "；对照题目三个字符串。")

✅ 课堂练习3：停止原因与执行时刻通过。


<details><summary>独立完成后查看参考要点</summary>

a complete；b step_limit；c repeated_call。c没有预算问题，不能只看调用次数；b即使工具只读，也按本课策略不启动新调用。

</details>

### 3.4 哪些边界已经实现，哪些还没有？
- 已覆盖：固定只读数据、名称白名单、参数校验、配对回传、有限模型请求、重复停止、工具失败不冒充查无记录。
- 不提供：任意Shell、读私人文件、删除/下单/发邮件、把模型口头“用户同意”当授权。
- 人工确认：真正写操作应由应用验证人类授权，再用幂等键等设计防止重复副作用；本课不声称已经实现。
- 隔离：虚拟环境仅隔离依赖，不是安全沙箱；max_steps不等于工具墙钟超时。代码执行隔离在后续B篇展开。
- 工具结果中的“忽略规则”仍是数据，不升级成system。JSON和角色分离降低风险，不是彻底防注入。

## 4. 课后作业：机房设备助手
只修改agent_tools.py的3个TODO，不修改测试、教师校验或SDK接线。本课不要求再写编码、排序或通用类型判断；核心是权限、消息配对和循环停止。

三个任务使用同一组**机房**对象，与图书馆示例、园区课堂练习不同。工具本体由调用方/测试提供，学生代码不查真实校园系统。

### 先看输入长什么样
```python
question = "北校区机房有多少台电脑？"  # str，用户提供
call = {
    "id": "device-call-1",
    "type": "function",
    "function": {
        "name": "get_devices",
        "arguments": '{"room_id":"room-a"}',  # str，不是dict
    },
}
```
上面是固定输入示例，不是真实模型实测。call是教师适配后的dict。type固定function；name来自模型请求。调用ID在整个输入轨迹中唯一，字段类型由适配层保证；学生不再实现SDK解析和通用类型守卫。

registry是dict[str, Callable]，例如把"find_room"映射到接受campus字符串的函数，把"get_devices"映射到接受room_id字符串的函数；值是函数，不是结果。Callable来自collections.abc，表示可调用对象，本课用普通函数。allowed_names是set[str]字符串集合，表示调用方明确批准的工具名；集合用于成员检查，不决定消息顺序。本课只批准这两个只读名称；注册表中仍可能放有未批准的测试工具，用来验证不能只检查是否注册。

教师validate_arguments已支持find_room的campus和get_devices的room_id。它返回新的参数dict或抛ValueError；工具本体返回可JSON序列化的dict（例如{"room_id":"room-a"}或{"count":20}），也可能抛异常。输入由调用方保证这些形状，不要求你设计新Schema。

### 4.1 execute_call(call, registry, allowed_names)
输出为本课工具结果dict，以下场景均须支持：
| 场景 | 输出 | 是否调用真实工具函数 |
|---|---|---|
| name不在registry | {"ok": False, "error": "unknown_tool"} | 否 |
| name已注册但未获准 | {"ok": False, "error": "not_allowed"} | 否 |
| 教师validate_arguments拒绝参数 | {"ok": False, "error": "bad_arguments"} | 否 |
| 获准且参数合格，工具正常返回data | {"ok": True, "data": data} | 恰好一次 |
| 工具本体抛TimeoutError | {"ok": False, "error": "tool_timeout"} | 已尝试一次，不重试 |
| 工具本体抛其他Exception | {"ok": False, "error": "tool_failed"} | 已尝试一次，不重试 |

如果名称既未知又未批准，归unknown_tool。校验阶段的ValueError是bad_arguments；**工具本体自己抛ValueError则属于tool_failed**，不能混淆两个阶段。异常全文不放进结果。正常返回{"found": False}仍是ok=True的数据，不是tool_failed。不得修改call、不得eval、不按globals找函数。

这是对1.1/1.3的迁移，不要求学习者完成教师底层的JSON类型/长度检查。

### 4.2 append_tool_turn(messages, assistant_message, outcomes)
目标：构造下一次要交给模型的消息快照，不联网。
- messages：list[dict]，调用方提供的已有消息。
- assistant_message：dict，本次模型提出工具调用的完整消息，tool_calls非空。
- outcomes：dict[str, dict]，程序处理得到的结果，**键是调用ID**，每个请求ID恰好有一个结果。不是结果列表，也不是按工具名称索引。

具体输入示例：
```python
messages = [
    {"role": "system", "content": AGENT_RULE},
    {"role": "user", "content": "room-a设备？"},
]
assistant_message = {"role": "assistant", "content": None, "tool_calls": [call]}
outcomes = {
    "device-call-1": {"ok": True, "data": {"count": 20}}
}
```
输出list[dict]：原有消息在前，接完整assistant_message，再按其tool_calls原顺序，给每个请求追加一条tool消息。
每条tool消息恰好role="tool"、tool_call_id=该请求id、content=该ID结果的JSON字符串（ensure_ascii=False，不加缩进）。outcomes的键顺序不影响回传顺序，失败/拒绝结果也要回传。

返回消息及嵌套tool_calls须独立复制，不能修改或共享输入中的可变字典/列表；outcomes也不修改。输入保证ID配套，不要求你再处理缺失键。不能漏掉原assistant请求，也不能只回第一个工具结果。

### 4.3 run_agent(question, ask, registry, allowed_names, max_steps=4)
它是本课协调函数。**hits不在本题参数里；ask也不是上课固定两次请求的observe_roundtrip。**
- question：str，用户问题。
- ask：Callable，由调用方传入的函数，接收一个消息快照list[dict]，返回规范assistant字典。它已经完成SDK协议检查，零工具时content必为非空字符串；有工具时content可为None或字符串、tool_calls为非空列表。测试里它不联网。
- registry/allowed_names：同函数1。
- max_steps：调用方提供的非负int，默认4；0允许，None不允许。不要求通用类型检查。

初始消息恰好system内容AGENT_RULE、user内容question。每次把独立消息快照交给ask；不可把模型输出改为系统规则。执行与消息回传复用函数1和2，不重新复制一套逻辑。

不抛异常时，返回结果包含下面这些字段。以“不需要工具，直接问好”为例：
```python
{
    "status": "complete",
    "answer": "你好",
    "messages": [
        {"role": "system", "content": AGENT_RULE},
        {"role": "user", "content": "你好"},
        {"role": "assistant", "content": "你好", "tool_calls": []},
    ],
    "model_calls": 1,
    "tool_calls": 0,
}
```
messages不是只有最终答案，而是system/user与各完整工具轮次。model_calls数ask正常返回的次数；**tool_calls数调用execute_call处理的请求数，包含被拒绝/失败的请求，不是成功执行工具数**。

| 场景 | 本课要求 |
|---|---|
| ask返回无工具的最终文字 | complete，answer为原content，轨迹末尾保留该assistant消息 |
| 问模型预算已耗尽 | step_limit，answer=""，不再请求 |
| 最后一次ask仍提出工具 | step_limit，answer=""，这批不调用execute_call、不追加缺结果半轮 |
| 仍有预算，但本批存在曾处理过/同批重复的操作签名 | repeated_call，answer=""，整批不处理、不追加半轮 |
| 仍有预算且不是重复 | 每个请求处理一次，所有结果按ID回传后才允许继续问模型 |
| ask抛异常 | 自然向调用方传播，不包装成成功/不知道，不重试 |

优先级：最终文字可在最后一次正常接受；若最后一次仍要工具，先按step_limit处理，不再判断重复。非最后一次再用operation_key检查整批重复，不能先执行一半才发现重复。每个曾处理的签名（包括工具拒绝与失败）计入记录；工具参数内容变化时不是重复。

例：两步依赖＋最终回答需要3次模型调用；max_steps=2时只能处理第一批工具，第二次若还要求工具就停止，不会偷偷加到3。多请求批次按原顺序逐个处理，不要求并发。不添加跨进程缓存或写操作幂等功能。

### 4.4 验收命令与边界
在本课目录：
```sh
python -m pytest -q test_agent_tools.py
```
共**23项核心契约测试，0项通用类型边界题**。只想检查一个函数：
```sh
python -m pytest -q test_agent_tools.py -k execute
python -m pytest -q test_agent_tools.py -k append
python -m pytest -q test_agent_tools.py -k run
```
初始TODO未写时应因NotImplementedError失败；这不代表环境坏了。教师底层另有12项test_tool_support.py，不冒充学生作业完成。创建时参考实现在仓库外验证，不在作业骨架中预填。

这组测试证明可复现的权限/配对/停止规则，不证明模型永远选择正确工具，也不证明最终文字的事实正确。参照第06课，真实质量仍需逐项核查。

## 5. 理解题：先凭记忆填写字母，再检测
不根据选项长度猜答案，每题选项均为11个字符。

1. 模型返回了工具调用请求，谁负责真正执行？
   - A：模型在远程直接执行函数
   - B：应用先检查许可再执行它
   - C：工具说明自行启动该函数
   - D：请求到达即自动执行函数

2. 响应里的function.arguments在本课怎样使用？
   - A：作为字典直接读取其字段
   - B：根据工具名称直接传进去
   - C：作为系统规则直接发回去
   - D：先解析并校验再用于调用

3. tool消息的tool_call_id应当取自哪里？
   - A：本次用户问题的原始编号
   - B：对应工具名称的注册编号
   - C：对应工具请求的原始编号
   - D：本次检索原文的资料编号

4. 本课最后一次模型请求仍提出新工具，应怎么处理？
   - A：停止且不启动本批新工具
   - B：执行后再多请求一次模型
   - C：执行后作为最终答案返回
   - D：修改预算后继续执行工具

5. 模型换新ID，重复请求同名同参数工具，应怎么处理？
   - A：名称相同便拒绝所有调用
   - B：改成直接展示先前的结果
   - C：只要编号不同就继续执行
   - D：识别重复并按本课约定停

In [12]:
quiz_answers = {"q1": "B", "q2": "D", "q3": "C", "q4": "A", "q5": "D"}

In [13]:
try:
    assert isinstance(quiz_answers, dict), "按q1至q5填写字母字典。"
    _check_correct = ["B","D","C","A","D"]
    _check_explanations = ["工具请求只是提议，应用必须检查注册表、许可和参数后才执行。","arguments是JSON字符串，教师校验函数返回参数字典后才能调用真实函数。","tool_call_id复制对应工具请求的id，不是工具名、用户编号或RAG资料ID。","预算包括最终回答请求，本课最后一步仍要工具时停止，不执行无法再交给模型使用的新工具。","重复按工具名和参数内容识别，不按调用ID；不同参数仍是新操作。"]
    _check_failed = 0
    for _check_key, _check_expected, _check_explanation in zip(
        ["q1", "q2", "q3", "q4", "q5"], _check_correct, _check_explanations
    ):
        _check_answer = quiz_answers.get(_check_key, "")
        if _check_answer not in ("A", "B", "C", "D"):
            print(f"❌ {_check_key}：先填A/B/C/D。")
            _check_failed += 1
        elif _check_answer != _check_expected:
            print(f"❌ {_check_key}：{_check_explanation}")
            _check_failed += 1
    if not _check_failed:
        print("✅ 5道理解题全部通过。")
except NameError:
    print("❌ 先运行quiz_answers答题格。")
except AssertionError as _check_error:
    print("❌", _check_error)
except Exception as _check_error:
    print("❌ 理解题结构不符：", type(_check_error).__name__, "；按题号填字母，不改检测代码。")

✅ 5道理解题全部通过。


## 6. 明天再做：关掉讲义，回忆一次工具闭环
**第二天再做，不是现在连刷。** 新建临时草稿、清空草稿答案，凭记忆重写；不清空正式Notebook答案。
1. 从一条用户问题画出五个角色/时刻：模型提出、应用执行、结果回传、模型继续、程序停止。
2. 自己举一个“工具定义不是执行授权”的例子，再写一条包含tool_call_id的工具结果。
3. 凭记忆写只读循环草图，说明max_steps=1遇到工具请求时为什么停止。
4. 混合复习第06课：工具查无记录、工具执行失败、模型请求失败有什么不同？为什么都不能随意说成no_answer？
5. 针对第四课副本遗留，在新草稿中用嵌套tool_calls证明浅复制/深复制差异；原正式作答不覆盖。

写不出来时只回看对应小节，再关掉重写。请把隔天表现告诉我，创建文件/今天读顺不算长期留存。

本课速查：[速查_工具调用.md](速查_工具调用.md) · [协议一页图](../reference/0004-tools-protocol.html)。
下一课才把RAG与工具流程接进后端；本次不提前创建第08课文件。